### 3.1.6. Problem Classification

$$
\min_{\mathbf{x}\in\mathcal{X}}\; f(\mathbf{x})
\quad\text{subject to}\quad
g_i(\mathbf{x}) \le 0,\;\; h_j(\mathbf{x}) = 0.
$$

$$
\begin{array}{ll}
\textbf{variables:} & \mathcal{X}\subseteq\mathbb{R}^n \ (\text{continuous}) \quad\text{or}\quad \mathcal{X}\subseteq\mathbb{Z}^n \ (\text{discrete}) \\[2pt]
\textbf{functions:} & \text{linear} \;\subset\; \text{quadratic} \;\subset\; \text{nonlinear} \\[2pt]
\textbf{curvature:} & \text{convex} \quad\text{vs.}\quad \text{nonconvex}
\end{array}
$$

**Explanation:**

Optimization problems are classified along a few structural axes, and the class a problem falls into determines both which optimality guarantees hold and which algorithms apply. The three most important axes are the type of the decision variables (continuous over $\mathbb{R}^n$ versus discrete over $\mathbb{Z}^n$), the algebraic form of the objective and constraint functions (linear, quadratic, or general nonlinear), and the curvature of the problem (convex versus nonconvex).

These axes name the rest of the track. Continuous problems with linear or quadratic structure are the [convex problem classes](../04_Convex_Problem_Classes/01_linear_programming.ipynb) LP and QP; general smooth continuous problems are [nonlinear programs](../05_Nonlinear_Programming/01_general_nonlinear_programs.ipynb); integer variables give [discrete and combinatorial](../06_Discrete_and_Combinatorial_Optimization/01_integer_programming.ipynb) problems. The single most consequential distinction is convexity: for a convex problem every [local optimum is global](./05_local_and_global_optima.ipynb), whereas a nonconvex problem may have many local optima and no efficient guarantee of finding the best one.

**Intuition:**

A convex objective has a single bowl-shaped basin, while a nonconvex one has several — the classification predicts whether a local method can be trusted to find the global solution.

![Convex versus nonconvex functions](../../Figures/030212_tordesillas_convex_vs_nonconvex_functions.png)

**Numerical Example:**

Classify three objectives in $\mathbf{x}=(x_1,x_2)$ by inspecting their total polynomial degree and their Hessian:
$$
f_1 = 2x_1 + 3x_2,\qquad
f_2 = 2x_1^2 + x_2^2 + x_1 x_2,\qquad
f_3 = x_1^4 - 3x_1^2 + x_2^2.
$$

- $f_1$ has degree $1$, so its Hessian is the zero matrix — **linear**, and trivially **convex**.
- $f_2$ has degree $2$ with constant Hessian $\begin{bmatrix}4 & 1\\ 1 & 2\end{bmatrix}$, whose eigenvalues $\tfrac{6\pm\sqrt{5}}{2}$ are positive — **quadratic** and **convex**.
- $f_3$ has degree $4$, so it is **nonlinear**; its Hessian $\begin{bmatrix}12x_1^2-6 & 0\\ 0 & 2\end{bmatrix}$ is indefinite for small $x_1$ — **nonconvex**.

The degree fixes the function form and the Hessian fixes the curvature, which together place each problem in its class.

In [1]:
import sympy as sp

x_1, x_2 = sp.symbols("x_1 x_2", real=True)
variables = [x_1, x_2]

def classify(objective):
    degree = sp.Poly(objective, *variables).total_degree()
    form = {0: "constant", 1: "linear", 2: "quadratic"}.get(degree, "nonlinear")
    hessian = sp.hessian(objective, variables)
    if degree <= 2:
        eigenvalues = list(hessian.eigenvals())
        convex = all(eigenvalue.is_real and eigenvalue >= 0 for eigenvalue in eigenvalues)
    else:
        convex = False
    return form, "convex" if convex else "nonconvex"

objectives = {"f_1": 2*x_1 + 3*x_2,
              "f_2": 2*x_1**2 + x_2**2 + x_1*x_2,
              "f_3": x_1**4 - 3*x_1**2 + x_2**2}

for name, objective in objectives.items():
    form, curvature = classify(objective)
    print(f"{name}: {form}, {curvature}")

f_1: linear, convex
f_2: quadratic, convex
f_3: nonlinear, nonconvex


**Equivalent casadi implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)

def classify(objective):
    hessian, _ = ca.hessian(objective, decision_variable)
    quadratic_or_less = not ca.depends_on(hessian, decision_variable)
    if not quadratic_or_less:
        return "nonlinear", "nonconvex"
    constant_hessian = ca.evalf(hessian)
    form = "linear" if float(ca.norm_1(constant_hessian)) == 0 else "quadratic"
    eigenvalues = ca.eig_symbolic(ca.SX(constant_hessian))
    convex = all(float(eigenvalues[i]) >= -1e-9 for i in range(eigenvalues.numel()))
    return form, "convex" if convex else "nonconvex"

x_1, x_2 = decision_variable[0], decision_variable[1]
objectives = {"f_1": 2*x_1 + 3*x_2,
              "f_2": 2*x_1**2 + x_2**2 + x_1*x_2,
              "f_3": x_1**4 - 3*x_1**2 + x_2**2}

for name, objective in objectives.items():
    form, curvature = classify(objective)
    print(f"{name}: {form}, {curvature}")

f_1: linear, convex
f_2: quadratic, convex
f_3: nonlinear, nonconvex


**References:**

[📘 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*, Ch. 1. Cambridge University Press.](https://web.stanford.edu/~boyd/cvxbook/)  
[📘 Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.), Ch. 1. Springer.](https://doi.org/10.1007/978-0-387-40065-5)

---

[⬅️ Previous: Local and Global Optima](./05_local_and_global_optima.ipynb) | [Next: Unconstrained Optimization ➡️](../02_Continuous_Optimization/01_unconstrained_optimization.ipynb)